# Publication figures from saved results (no recomputation)

This notebook only **reads** what the pipeline notebooks already saved on Google Drive
(`MyDrive/KG_Strong_Pipeline/study_A`, `study_B`, `study_C`) and draws publication-quality figures:

* **Knowledge-graph figures** — ego network of one asset with typed relations, community meta-graph,
  relation multiplex (one panel per relation), class-to-class connectivity, degree distributions,
  a 2-D map of the text embeddings, structural exposure.
* **Result figures** — seed distributions, precision–recall curves, model ladder, GNN − pair-only,
  cold start and community persistence, the known-market test, external data, feature attribution,
  Task-1 seeds and portfolios.

Every figure is saved as vector PDF and 600-dpi PNG to `MyDrive/KG_Strong_Pipeline/figures_q1/`.
A figure whose inputs are missing is skipped with a message; nothing is recomputed.

**Run (Colab):** `Runtime → Run all` (CPU is enough). Optional: change `FOCAL_SYMBOL` in the settings cell.

In [ ]:
# CELL F0 — settings and paths
import os, sys, json, glob, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

STUDIES = os.environ.get("KG_FIG_STUDIES", "A,B").split(",")   # which studies to draw
FOCAL_SYMBOL = None          # e.g. "SPY" or "AAPL"; None = the highest-degree ETF
GRAPH_STUDY = None           # study whose graph is drawn; None = first study with a saved graph
SAMPLE_NODES = 160           # nodes in the relation-multiplex figure
TSNE_MAX = 3000              # assets in the embedding map
DPI = 600

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/KG_Strong_Pipeline")
except Exception:
    BASE = Path(os.environ.get("KG_BASE", Path.cwd() / "KG_Strong_Pipeline"))
OUT = BASE / "figures_q1"
OUT.mkdir(parents=True, exist_ok=True)
print("reading from", BASE, "| writing to", OUT)

Mounted at /content/drive
reading from /content/drive/MyDrive/KG_Strong_Pipeline | writing to /content/drive/MyDrive/KG_Strong_Pipeline/figures_q1


In [ ]:
# CELL F1 — style, palette and helpers
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Patch
from matplotlib.lines import Line2D

# categorical palette (validated for colour-vision deficiency on a light surface), fixed order
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e6e5df"
CLASSES = ["Equity", "ETF", "Fund", "Crypto", "Index", "Currency"]
CLASS_COL = dict(zip(CLASSES, PAL))
REL_ORDER = ["semantic_similarity", "same_sector", "same_fund_category", "same_country", "same_exchange", "same_currency"]
REL_LAB = {"semantic_similarity": "Semantic kNN", "same_sector": "Sector", "same_fund_category": "Fund category",
           "same_country": "Country", "same_exchange": "Exchange", "same_currency": "Currency"}
REL_COL = dict(zip(REL_ORDER, PAL))
STUDY_COL = {"A": PAL[0], "B": PAL[1], "C": PAL[2]}
FOLD_LAB = {"F1": "test 2023–24", "F2": "test 2024–25"}
FOUR = {("A", "F1"): (PAL[0], "o"), ("A", "F2"): (PAL[1], "D"), ("B", "F1"): (PAL[2], "s"), ("B", "F2"): (PAL[3], "^")}

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "pdf.fonttype": 42, "ps.fonttype": 42})


def style(ax, grid="x"):
    if grid:
        ax.grid(axis=grid, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)


def panel(ax, letter, title):
    ax.set_title(f"({letter})  {title}", loc="left", color=INK)


def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(OUT / f"{name}.png", dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print("  saved", name)


def res(study, name):
    p = BASE / f"study_{study}" / "results" / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else None


def stage(study, name):
    files = sorted(glob.glob(str(BASE / f"study_{study}" / "stages" / f"{name}__*.pkl")), key=os.path.getmtime)
    if not files:
        return None
    with open(files[-1], "rb") as fh:
        return pickle.load(fh)


def figure(fn):
    """Run one figure function; a missing input skips the figure instead of stopping the notebook."""
    try:
        fn()
    except Exception as e:
        print(f"  skipped {fn.__name__}: {type(e).__name__}: {e}")


def class_legend(ax, classes, **kw):
    ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=CLASS_COL[c], mec="white", ms=7, label=c) for c in classes],
              frameon=False, **kw)

In [ ]:
# CELL F2 — load the graph of one study (universe, relations, communities, embeddings)
import html
import networkx as nx
G_ASSETS = G_REL = G_COMM = G_X = None
if GRAPH_STUDY is None:
    GRAPH_STUDY = next((s for s in STUDIES if list((BASE / f"study_{s}" / "stages").glob("universe__*.pkl"))), STUDIES[0])
u = stage(GRAPH_STUDY, "universe")
if u is not None:
    G_ASSETS = u[0] if isinstance(u, tuple) else u
    G_REL = stage(GRAPH_STUDY, "graph")
    c = stage(GRAPH_STUDY, "communities")
    G_COMM = c[0] if isinstance(c, tuple) else c
    G_X = stage(GRAPH_STUDY, "embeddings")
    N = len(G_ASSETS)
    ATYPE = G_ASSETS.asset_type.to_numpy()
    G = nx.Graph(); G.add_nodes_from(range(N))
    for r, E in G_REL.items():
        for a, b in np.asarray(E):
            if G.has_edge(a, b):
                G[a][b]["rels"].add(r)
            else:
                G.add_edge(int(a), int(b), rels={r})
    print(f"Study {GRAPH_STUDY}: {N:,} assets, {G.number_of_edges():,} union edges, "
          f"{len(set(G_COMM)) if G_COMM is not None else 0} communities")
else:
    print(f"no saved universe for Study {GRAPH_STUDY}; knowledge-graph figures will be skipped")

Study A: 2,734 assets, 168,237 union edges, 8 communities


## Knowledge-graph figures

In [ ]:
# CELL F3 — KG1: ego network of one asset with typed relations
def kg_ego_network():
    if FOCAL_SYMBOL is not None and (G_ASSETS.symbol == FOCAL_SYMBOL).any():
        focal = int(np.where(G_ASSETS.symbol.to_numpy() == FOCAL_SYMBOL)[0][0])
    else:
        etf = [i for i in range(N) if ATYPE[i] == "ETF"]
        focal = max(etf or range(N), key=lambda i: G.degree(i))
    nb = list(G.neighbors(focal))
    rng = np.random.default_rng(7)
    # keep every cross-asset neighbour and a sample of the rest, so the figure stays readable
    cross = [j for j in nb if ATYPE[j] != ATYPE[focal]]
    same = [j for j in nb if ATYPE[j] == ATYPE[focal]]
    keep = cross[:25] + list(rng.choice(same, min(len(same), 35 - min(len(cross), 25)), replace=False)) if same else cross[:35]
    H = G.subgraph([focal] + [int(k) for k in keep]).copy()
    pos = nx.spring_layout(H, seed=3, k=0.9 / np.sqrt(max(len(H), 1)))
    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    for r in REL_ORDER:
        el = [(a, b) for a, b, d in H.edges(data=True) if r in d["rels"]]
        nx.draw_networkx_edges(H, pos, edgelist=el, edge_color=REL_COL[r], width=1.1, alpha=0.55, ax=ax)
    nodes = list(H.nodes())
    nx.draw_networkx_nodes(H, pos, nodelist=nodes, node_color=[CLASS_COL.get(ATYPE[n], MUTED) for n in nodes],
                           node_size=[420 if n == focal else 90 for n in nodes], edgecolors="white", linewidths=1.0, ax=ax)
    labels = {n: G_ASSETS.symbol.iloc[n] for n in nodes if n == focal or H.degree(n) >= np.percentile([H.degree(m) for m in nodes], 80)}
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=6.2, font_color=INK, ax=ax,
                            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))
    ax.axis("off")
    rel_h = [Line2D([0], [0], color=REL_COL[r], lw=2, label=REL_LAB[r]) for r in REL_ORDER
             if any(r in d["rels"] for _, _, d in H.edges(data=True))]
    cls_h = [Line2D([0], [0], marker="o", ls="", mfc=CLASS_COL[c], mec="white", ms=7, label=c)
             for c in CLASSES if c in {ATYPE[n] for n in nodes}]
    ax.legend(handles=cls_h + rel_h, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), title="Asset class / relation",
              title_fontsize=7.5)
    ax.set_title(f"Ego network of {G_ASSETS.symbol.iloc[focal]} ({ATYPE[focal]}, {html.unescape(str(G_ASSETS.name.iloc[focal]))[:40]}) — "
                 f"{G.degree(focal)} neighbours, {len(nodes) - 1} shown", loc="left", color=INK)
    save(fig, "kg1_ego_network")

In [ ]:
# CELL F4 — KG2: community meta-graph (pie nodes show asset-class composition)
def kg_community_metagraph():
    comm = np.asarray(G_COMM); K = comm.max() + 1
    W = np.zeros((K, K))
    for a, b in G.edges():
        W[comm[a], comm[b]] += 1; W[comm[b], comm[a]] += 1 if comm[a] != comm[b] else 0
    M = nx.Graph()
    sizes = np.bincount(comm, minlength=K)
    for k in range(K):
        M.add_node(k)
    for i in range(K):
        for j in range(i + 1, K):
            if W[i, j] > 0:
                M.add_edge(i, j, w=W[i, j])
    pos = nx.spring_layout(M, seed=11, weight="w", k=1.6)
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    wmax = max([d["w"] for _, _, d in M.edges(data=True)] or [1])
    for a, b, d in M.edges(data=True):
        ax.plot(*zip(pos[a], pos[b]), color=MUTED, lw=0.4 + 5 * d["w"] / wmax, alpha=0.35, zorder=1)
    rmax = 0.18
    for k in range(K):
        r = rmax * np.sqrt(sizes[k] / sizes.max()) + 0.03
        comp = pd.Series(ATYPE[comm == k]).value_counts(normalize=True)
        start = 90.0
        for cls in CLASSES:
            if cls in comp:
                ang = 360 * comp[cls]
                ax.add_patch(Wedge(pos[k], r, start, start + ang, fc=CLASS_COL[cls], ec="white", lw=0.8, zorder=2))
                start += ang
        ax.text(pos[k][0], pos[k][1] - r - 0.05, f"C{k}  n={sizes[k]:,}", ha="center", va="top", fontsize=6.8, color=INK)
    ax.set_aspect("equal"); ax.autoscale_view(); ax.axis("off")
    class_legend(ax, [c for c in CLASSES if c in set(ATYPE)], loc="upper left", bbox_to_anchor=(1.0, 1.0))
    ax.set_title(f"Leiden communities of the 2021 graph (Study {GRAPH_STUDY}): node area ∝ assets, "
                 "edge width ∝ edges between communities", loc="left", color=INK)
    save(fig, "kg2_community_metagraph")

In [ ]:
# CELL F5 — KG3: relation multiplex — one panel per relation on a common layout
def kg_relation_multiplex():
    rng = np.random.default_rng(5)
    per = max(SAMPLE_NODES // len(set(ATYPE)), 5)
    sample = []
    for c in CLASSES:
        idx = np.where(ATYPE == c)[0]
        if len(idx):
            sample += list(rng.choice(idx, min(per, len(idx)), replace=False))
    H = G.subgraph([int(i) for i in sample]).copy()
    pos = nx.spring_layout(H, seed=2, k=1.2 / np.sqrt(len(H)))
    fig, axes = plt.subplots(2, 3, figsize=(7.4, 5.2))
    for ax, r, letter in zip(axes.ravel(), REL_ORDER, "abcdef"):
        el = [(a, b) for a, b, d in H.edges(data=True) if r in d["rels"]]
        nx.draw_networkx_edges(H, pos, edgelist=el, edge_color=REL_COL[r], width=0.6, alpha=0.6, ax=ax)
        nx.draw_networkx_nodes(H, pos, node_color=[CLASS_COL.get(ATYPE[n], MUTED) for n in H.nodes()], node_size=9,
                               edgecolors="none", ax=ax)
        ax.set_title(f"({letter})  {REL_LAB[r]} — {len(el)} edges", loc="left", fontsize=8, color=INK); ax.axis("off")
    fig.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=CLASS_COL[c], mec="white", ms=6, label=c)
                        for c in CLASSES if c in set(ATYPE)], frameon=False, loc="upper center", ncol=6)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    save(fig, "kg3_relation_multiplex")

In [ ]:
# CELL F6 — KG4: class-to-class connectivity (share of each class's edges going to each class)
def kg_class_connectivity():
    present = [c for c in CLASSES if c in set(ATYPE)]
    idx = {c: k for k, c in enumerate(present)}
    mats = {}
    for r in ["union"] + REL_ORDER:
        M = np.zeros((len(present), len(present)))
        E = np.array(list(G.edges())) if r == "union" else np.asarray(G_REL.get(r, np.zeros((0, 2))))
        for a, b in E:
            M[idx[ATYPE[a]], idx[ATYPE[b]]] += 1; M[idx[ATYPE[b]], idx[ATYPE[a]]] += 1
        mats[r] = M / np.maximum(M.sum(1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), gridspec_kw=dict(width_ratios=[1.15, 1]))
    im = axes[0].imshow(mats["union"], cmap="Blues", vmin=0, vmax=1)
    axes[0].set_xticks(range(len(present))); axes[0].set_xticklabels(present, rotation=45, ha="right")
    axes[0].set_yticks(range(len(present))); axes[0].set_yticklabels(present)
    for i in range(len(present)):
        for j in range(len(present)):
            v = mats["union"][i, j]
            axes[0].text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.55 else INK)
    panel(axes[0], "a", "Union graph: where each class's edges go")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.03)
    cross = []
    for r in REL_ORDER:
        E = np.asarray(G_REL.get(r, np.zeros((0, 2))))
        cross.append(100 * float((ATYPE[E[:, 0]] != ATYPE[E[:, 1]]).mean()) if len(E) else 0.0)
    y = np.arange(len(REL_ORDER))[::-1]
    axes[1].barh(y, cross, color=[REL_COL[r] for r in REL_ORDER], height=0.6)
    for yy, v in zip(y, cross):
        axes[1].text(v + 1, yy, f"{v:.1f}%", va="center", fontsize=7, color=INK)
    axes[1].set_yticks(y); axes[1].set_yticklabels([REL_LAB[r] for r in REL_ORDER]); axes[1].set_xlim(0, max(cross + [10]) * 1.25)
    axes[1].set_xlabel("Edges joining different asset classes (%)"); panel(axes[1], "b", "Cross-asset share by relation"); style(axes[1])
    fig.tight_layout()
    save(fig, "kg4_class_connectivity")

In [ ]:
# CELL F7 — KG5: degree distributions (complementary CDF, log–log)
def kg_degree_ccdf():
    fig, ax = plt.subplots(figsize=(4.8, 3.4))
    for r in ["union"] + REL_ORDER:
        if r == "union":
            deg = np.array([d for _, d in G.degree()])
        else:
            E = np.asarray(G_REL.get(r, np.zeros((0, 2))))
            deg = np.bincount(E.ravel(), minlength=N) if len(E) else np.zeros(N)
        deg = np.sort(deg[deg > 0])
        if len(deg) == 0:
            continue
        ccdf = 1 - np.arange(len(deg)) / len(deg)
        ax.loglog(deg, ccdf, lw=2.2 if r == "union" else 1.4, color=INK if r == "union" else REL_COL[r],
                  label="Union" if r == "union" else REL_LAB[r])
    ax.set_xlabel("Degree k"); ax.set_ylabel("P(degree ≥ k)"); style(ax, grid="both")
    ax.legend(frameon=False, loc="lower left")
    ax.set_title(f"Degree distributions by relation (Study {GRAPH_STUDY})", loc="left")
    save(fig, "kg5_degree_ccdf")

In [ ]:
# CELL F8 — KG6: 2-D map of the frozen MPNet description embeddings (t-SNE)
def kg_embedding_map():
    from sklearn.manifold import TSNE
    rng = np.random.default_rng(1)
    idx = np.arange(N) if N <= TSNE_MAX else np.sort(rng.choice(N, TSNE_MAX, replace=False))
    Y = TSNE(n_components=2, perplexity=30, init="pca", random_state=0, learning_rate="auto").fit_transform(np.asarray(G_X)[idx])
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    for c in CLASSES:
        m = ATYPE[idx] == c
        if m.any():
            ax.scatter(Y[m, 0], Y[m, 1], s=5, color=CLASS_COL[c], alpha=0.75, label=c, edgecolors="none")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.legend(frameon=False, markerscale=2.5, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    ax.set_title(f"t-SNE of description embeddings (Study {GRAPH_STUDY}, {len(idx):,} assets)", loc="left")
    save(fig, "kg6_embedding_map")

In [ ]:
# CELL F9 — KG7: structural exposure (relation-balanced personalised PageRank mass, class → class)
def kg_exposure():
    M = res(GRAPH_STUDY, "structural_exposure_matrix")
    M = M.set_index("source_class")
    cols = [c for c in CLASSES if c in M.columns]; rows = [c for c in CLASSES if c in M.index]
    V = M.loc[rows, cols].to_numpy()
    fig, ax = plt.subplots(figsize=(4.6, 3.6))
    im = ax.imshow(V, cmap="Blues", vmin=0, vmax=max(0.5, V.max()))
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
    for i in range(len(rows)):
        for j in range(len(cols)):
            ax.text(j, i, f"{V[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if V[i, j] > 0.4 else INK)
    ax.set_xlabel("Destination class"); ax.set_ylabel("Source asset (most central of its class)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="Personalised PageRank mass")
    ax.set_title("Structural exposure across asset classes", loc="left")
    save(fig, "kg7_structural_exposure")


if G_ASSETS is not None:
    for fn in (kg_ego_network, kg_community_metagraph, kg_relation_multiplex, kg_class_connectivity, kg_degree_ccdf,
               kg_embedding_map, kg_exposure):
        figure(fn)

  saved kg1_ego_network
  saved kg2_community_metagraph
  saved kg3_relation_multiplex
  saved kg4_class_connectivity
  saved kg5_degree_ccdf
  saved kg6_embedding_map
  saved kg7_structural_exposure


## Result figures

In [ ]:
# CELL F10 — R1: AP of every run (one dot per seed) — Task 2, both folds and studies
MODELS = ["Text cosine", "Residual lagged correlation", "Lagged correlation", "Logistic", "XGBoost", "Pair-only",
          "GCN", "GraphSAGE", "GAT", "R-GCN", "Gated-RelGAT"]


def runs2(study):
    t = res(study, "task2_runs")
    if t is None:
        return None
    return t[(t.freq == "D") & (np.isclose(t.tau, 0.5)) & (t.lag == "lag")]


def r_seed_distribution():
    studies = [s for s in STUDIES if runs2(s) is not None]
    fig, axes = plt.subplots(1, len(studies), figsize=(3.7 * len(studies), 3.8), sharey=True, squeeze=False)
    rng = np.random.default_rng(0)
    for ax, s in zip(axes[0], studies):
        t = runs2(s)
        y0 = np.arange(len(MODELS))[::-1]
        for f, off, (col, mk) in (("F1", 0.17, FOUR[(s, "F1")]), ("F2", -0.17, FOUR[(s, "F2")])):
            for y, m in zip(y0, MODELS):
                v = t[(t.fold == f) & (t.model == m)].AP.to_numpy()
                if len(v):
                    ax.scatter(v, y + off + rng.uniform(-0.05, 0.05, len(v)), s=12, color=col, marker=mk, alpha=0.8,
                               edgecolors="white", linewidths=0.5, label=FOLD_LAB[f] if m == MODELS[0] else None)
        ax.set_yticks(y0); ax.set_yticklabels(MODELS); ax.set_xlabel("Locked-test AP (each dot one seed)")
        ax.set_title(f"Study {s}", loc="left"); style(ax)
    fig.legend(*axes[0][0].get_legend_handles_labels(), frameon=False, loc="upper center", ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    save(fig, "r1_task2_seed_distribution")

In [ ]:
# CELL F11 — R2: precision–recall curves from the saved test scores (seed 42)
def r_pr_curves():
    import torch
    from sklearn.metrics import precision_recall_curve, average_precision_score
    show = ["Lagged correlation", "Logistic", "Pair-only", "GAT", "Text cosine"]
    cols = dict(zip(show, [INK, PAL[0], PAL[1], PAL[2], PAL[3]]))
    studies = [s for s in STUDIES if runs2(s) is not None]
    fig, axes = plt.subplots(len(studies), 2, figsize=(7.2, 3.1 * len(studies)), squeeze=False)
    for i, s in enumerate(studies):
        t = runs2(s)
        for j, f in enumerate(("F1", "F2")):
            ax = axes[i][j]
            blocks = stage(s, f"task2_pairs_{f}_D")
            y = (np.asarray(blocks["test"]["target"]) > 0.5).astype(int)
            for m in show:
                r = t[(t.fold == f) & (t.model == m) & (t.seed == 42)]
                if not len(r):
                    continue
                path = BASE / f"study_{s}" / "runs" / f"{r.key.iloc[0]}.final.pt"
                if not path.exists():
                    continue
                o = torch.load(path, map_location="cpu", weights_only=False)
                sc = np.asarray(o["test"] if "test" in o else o["test_scores"])
                if len(sc) != len(y):
                    continue
                pr, rc, _ = precision_recall_curve(y, sc)
                ax.plot(rc, pr, color=cols[m], lw=1.6 if m != "Text cosine" else 1.1,
                        ls="--" if m == "Lagged correlation" else "-", label=f"{m} (AP {average_precision_score(y, sc):.3f})")
            ax.axhline(y.mean(), color=MUTED, lw=0.8, ls=":")
            ax.text(0.99, y.mean() + 0.02, f"base rate {y.mean():.3f}", ha="right", fontsize=6.5, color=MUTED)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
            panel(ax, "abcd"[2 * i + j], f"Study {s}, {FOLD_LAB[f]}"); style(ax, grid="both")
            ax.legend(frameon=False, fontsize=6.4, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    fig.tight_layout()
    save(fig, "r2_precision_recall")

In [ ]:
# CELL F12 — R3: GNN − pair-only (node-cluster bootstrap) and R4: Task-1 seeds
GNNS = ["GCN", "GraphSAGE", "GAT", "R-GCN", "Gated-RelGAT"]


def r_gnn_minus_pair():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharey=True)
    for ax, f in zip(axes, ("F1", "F2")):
        for k, s in enumerate(STUDIES):
            t = res(s, "task2_bootstrap_contrasts")
            if t is None:
                continue
            t = t[(t.freq == "D") & (t.fold == f) & (t.versus == "Pair-only") & t.model.isin(GNNS)].set_index("model").reindex(GNNS)
            y = np.arange(len(GNNS))[::-1] + (0.15 - 0.3 * k)
            ax.errorbar(t["diff"], y, xerr=[t["diff"] - t.ci_l, t.ci_u - t["diff"]], fmt="oD"[k % 2], color=STUDY_COL[s], ms=5,
                        lw=1.4, mec="white", mew=1, label=f"Study {s}")
        ax.axvline(0, color=MUTED, lw=0.8); ax.set_yticks(np.arange(len(GNNS))[::-1]); ax.set_yticklabels(GNNS)
        ax.set_xlabel("AP difference, GNN − pair-only (95% node-cluster CI)"); panel(ax, "ab"[f == "F2"], FOLD_LAB[f]); style(ax)
    fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc="upper right", ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    save(fig, "r3_gnn_minus_pair")


def r_task1_seeds():
    fig, ax = plt.subplots(figsize=(5.4, 2.9))
    order = ["R-GCN", "Gated-RelGAT", "GAT", "GCN", "GraphSAGE", "Logistic regression", "Text cosine"]
    rng = np.random.default_rng(1)
    for k, s in enumerate(STUDIES):
        t = res(s, "task1_runs")
        if t is None:
            continue
        for y, m in zip(np.arange(len(order))[::-1], order):
            v = t[t.model == m].AP.dropna().to_numpy()
            if len(v):
                ax.scatter(v, y + (0.15 - 0.3 * k) + rng.uniform(-0.04, 0.04, len(v)), s=14, color=STUDY_COL[s], marker="oD"[k % 2],
                           edgecolors="white", linewidths=0.5, label=f"Study {s}" if m == order[0] else None)
    ax.set_yticks(np.arange(len(order))[::-1]); ax.set_yticklabels(order); ax.set_xlabel("Task-1 test AP (each dot one seed)")
    ax.set_title("Structural reconstruction: ten seeds per GNN", loc="left"); style(ax)
    fig.legend(*ax.get_legend_handles_labels(), frameon=False, loc="upper right", ncol=2)
    save(fig, "r4_task1_seeds")

In [ ]:
# CELL F13 — R5: cold start and community persistence; R6: known-market test
def r_coldstart_community():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5))
    for k, s in enumerate(STUDIES):
        p = res(s, "prereg_contrasts"); c = res(s, "community_validation")
        if p is not None:
            g = p[(p.hyp == "H1") & (p.primary == True)].set_index("fold").reindex(["F1", "F2"])
            y = np.array([1, 0]) + (0.12 - 0.24 * k)
            axes[0].errorbar(g["diff"], y, xerr=[g["diff"] - g.ci_l, g.ci_u - g["diff"]], fmt="oD"[k % 2], color=STUDY_COL[s], ms=5,
                             lw=1.4, mec="white", mew=1, label=f"Study {s}")
        if c is not None:
            c = c.set_index("window").reindex(["t3", "t4"])
            y = np.array([1, 0]) + (0.12 - 0.24 * k)
            axes[1].errorbar(c["diff"], y, xerr=[c["diff"] - c.ci_l, c.ci_u - c["diff"]], fmt="oD"[k % 2], color=STUDY_COL[s], ms=5,
                             lw=1.4, mec="white", mew=1)
    axes[0].set_yticks([1, 0]); axes[0].set_yticklabels(["2023–24", "2024–25"]); axes[0].set_xlabel("AP gain over class-only model (95% CI)")
    panel(axes[0], "a", "Cold start: text + relations, no returns")
    axes[1].set_yticks([1, 0]); axes[1].set_yticklabels(["2023–24", "2024–25"])
    axes[1].set_xlabel("Within − across-community correlation (95% CI)"); panel(axes[1], "b", "2021 communities persist")
    for ax in axes:
        ax.axvline(0, color=MUTED, lw=0.8); ax.set_ylim(-0.5, 1.5); style(ax)
    fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc="upper right", ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    save(fig, "r5_coldstart_community")


H5_ORDER = ["Logistic (no propagation)", "Text cosine", "Text-neighbour propagation (no learning)", "KG propagation (no learning)",
            "Logistic", "Pair-only", "GAT", "R-GCN"]
H5_LAB = {"Logistic (no propagation)": "Logistic, text + relations only", "Text cosine": "Description cosine",
          "Text-neighbour propagation (no learning)": "Text-neighbour propagation", "KG propagation (no learning)": "KG-neighbour propagation",
          "Logistic": "Logistic + propagation", "Pair-only": "Pair-only + propagation", "GAT": "GAT + propagation", "R-GCN": "R-GCN + propagation"}


def r_known_market():
    studies = [s for s in STUDIES if res(s, "h5_summary") is not None]
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 3.1), gridspec_kw=dict(width_ratios=[1.25, 0.7, 1.0]))
    ym = np.arange(len(H5_ORDER))[::-1]
    for k, s in enumerate(studies):
        t = res(s, "h5_summary").set_index(["fold", "model"])
        for j, f in enumerate(("F1", "F2")):
            col, mk = FOUR[(s, f)]
            axes[0].scatter([t.AP_mean.get((f, m), np.nan) for m in H5_ORDER], ym + 0.18 - 0.12 * (2 * k + j), color=col, marker=mk, s=28,
                            edgecolors="white", linewidths=1, label=f"Study {s}, {FOLD_LAB[f]}", zorder=3)
    axes[0].set_yticks(ym); axes[0].set_yticklabels([H5_LAB[m] for m in H5_ORDER]); axes[0].set_xlabel("Test AP, new–known pairs")
    panel(axes[0], "a", "A new asset joins a known market")
    for ax, cons, letter, title in ((axes[1], [("Logistic − Logistic (no propagation)", "Propagation")], "b", "Propagation gain"),
                                    (axes[2], [("GAT − Logistic", "GAT − logistic"), ("R-GCN − Logistic", "R-GCN − logistic")], "c", "Message passing")):
        rows = []
        for con, lab in cons:
            for s in studies:
                t = res(s, "h5_contrasts")
                for f in ("F1", "F2"):
                    r = t[t.contrast.str.startswith(con[:30]) & (t.fold == f)]
                    if len(r):
                        r = r.iloc[0]; rows.append((f"{lab}, {s} {f}", r["diff"], r.ci_l, r.ci_u, FOUR[(s, f)]))
        y = np.arange(len(rows))[::-1]
        for yy, (lab, d, l, u, (col, mk)) in zip(y, rows):
            ax.errorbar([d], [yy], xerr=[[d - l], [u - d]], fmt=mk, color=col, ms=5, lw=1.3, mec="white", mew=1)
        ax.axvline(0, color=MUTED, lw=0.8); ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows], fontsize=6.6)
        ax.set_xlabel("AP difference (95% CI)"); panel(ax, letter, title)
    for ax in axes:
        style(ax)
    fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc="upper center", ncol=4)
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    save(fig, "r6_known_market")

In [ ]:
# CELL F14 — R7: what the models use (feature-group permutation) and R8: portfolios
GROUPS = ["Lagged correlation (+ missing flag)", "Asset-class-pair indicators", "Description cosine", "Relation indicators", "Same asset class"]
GLAB = ["Lagged correlation", "Class pair", "Description", "Relations", "Same class"]


def r_attribution():
    studies = [s for s in STUDIES if res(s, "attribution_permutation") is not None]
    fig, axes = plt.subplots(1, len(studies), figsize=(3.7 * len(studies), 3.0), sharey=True, squeeze=False)
    for ax, s in zip(axes[0], studies):
        t = res(s, "attribution_permutation")
        models = [m for m in ["Pair-only", "GAT", "R-GCN"] if m in set(t.model)]
        cells = [(m, f) for m in models for f in ("F1", "F2")]
        V = np.array([[t[(t.model == m) & (t.fold == f) & (t.group == g)].AP_mean.mean() for (m, f) in cells] for g in GROUPS])
        im = ax.imshow(V, cmap="Blues", vmin=0, vmax=np.nanmax(V))
        ax.set_xticks(range(len(cells))); ax.set_xticklabels([f"{m}\n{f}" for m, f in cells], fontsize=6.5)
        ax.set_yticks(range(len(GROUPS))); ax.set_yticklabels(GLAB)
        for i in range(V.shape[0]):
            for j in range(V.shape[1]):
                ax.text(j, i, f"{V[i, j]:.3f}", ha="center", va="center", fontsize=6, color="white" if V[i, j] > 0.5 * np.nanmax(V) else INK)
        ax.set_title(f"Study {s}: AP lost when permuted", loc="left")
    fig.tight_layout()
    save(fig, "r7_feature_attribution")


def r_portfolios():
    sel = ["Lagged correlation", "Logistic", "XGBoost", "Pair-only", "GAT", "Text cosine"]
    fig, ax = plt.subplots(figsize=(7.2, 2.8))
    ys = np.arange(len(sel))[::-1]
    k = 0
    for s in STUDIES:
        t = res(s, "economic_portfolios")
        if t is None:
            continue
        for f in ("F1", "F2"):
            col, mk = FOUR.get((s, f), (PAL[k % 6], "o"))
            kk = 20 if (t.K == 20).any() else t.K.max()   # paper uses K = 20
            g = t[(t.K == kk) & (t.fold == f)].set_index("selector").reindex(sel)
            ax.scatter(100 * g.corr_pctile_vs_classmatched, ys + 0.24 - 0.16 * k, color=col, marker=mk, s=28, edgecolors="white",
                       linewidths=1, label=f"Study {s}, {FOLD_LAB[f]}", zorder=3)
            k += 1
    ax.axvline(5, color=MUTED, lw=0.8, ls=":"); ax.set_yticks(ys); ax.set_yticklabels(sel)
    ax.set_xlabel("Realised mean pairwise correlation: percentile among class-matched random portfolios (lower = better)")
    style(ax); fig.legend(*ax.get_legend_handles_labels(), frameon=False, loc="upper center", ncol=4)
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    save(fig, "r8_portfolios")

In [ ]:
# CELL F15 — R9: external relations (holdings, news co-mention): signal present, increment absent
def r_external():
    rows, cons = [], []
    for s in STUDIES:
        d = res(s, "external_details"); c = res(s, "external_contrasts")
        if d is not None and "positive_rate_with_link" in d:
            for _, r in d[d.ext == "E2"].iterrows():
                rows.append((f"Holdings, {s} {r.fold}", 100 * r.positive_rate_with_link, 100 * r.positive_rate_all))
        if c is not None:
            for hyp, lab in (("E2a", "Holdings, with lag"), ("E2b", "Holdings, cold start")):
                for f in ("F1", "F2"):
                    r = c[(c.hyp == hyp) & (c.fold == f) & (c.primary == True)]
                    if len(r):
                        r = r.iloc[0]; cons.append((f"{lab}, {s} {f}", r["diff"], r.ci_l, r.ci_u))
    cd = res("C", "Cplus_details"); cc = res("C", "Cplus_contrasts")
    if cd is not None:
        for _, r in cd[cd.part == "C5"].iterrows():
            rows.append((f"News co-mention, C {r.fold}", 100 * r.positive_rate_co_mentioned, 100 * r.positive_rate_other))
    if cc is not None:
        for hyp, lab in (("C5a", "Co-mention, with lag"), ("C5b", "Co-mention, cold start")):
            for f in ("F1", "F2"):
                r = cc[(cc.hyp == hyp) & (cc.fold == f) & (cc.primary == True)]
                if len(r):
                    r = r.iloc[0]; cons.append((f"{lab}, C {f}", r["diff"], r.ci_l, r.ci_u))
    if not rows and not cons:
        raise FileNotFoundError("no external results found")
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0))
    y = np.arange(len(rows))[::-1]
    for yy, (lab, linked, base) in zip(y, rows):
        axes[0].plot([base, linked], [yy, yy], color=GRID, lw=2.2, zorder=1)
    axes[0].scatter([r[2] for r in rows], y, color=MUTED, s=26, zorder=3, label="All / unlinked pairs", edgecolors="white")
    axes[0].scatter([r[1] for r in rows], y, color=PAL[0], s=30, zorder=3, label="Linked pairs", edgecolors="white")
    axes[0].set_yticks(y); axes[0].set_yticklabels([r[0] for r in rows]); axes[0].set_xlabel("Pairs with ρ > 0.5 (%)")
    panel(axes[0], "a", "Linked pairs co-move more often")
    y = np.arange(len(cons))[::-1]
    d = np.array([c[1] for c in cons]); lo = np.array([c[2] for c in cons]); hi = np.array([c[3] for c in cons])
    axes[1].errorbar(d, y, xerr=[d - lo, hi - d], fmt="o", color=PAL[1], ms=4.5, lw=1.3, mec="white", mew=1)
    axes[1].axvline(0, color=MUTED, lw=0.8); axes[1].set_yticks(y); axes[1].set_yticklabels([c[0] for c in cons], fontsize=6.6)
    axes[1].set_xlabel("AP gain (95% CI)"); panel(axes[1], "b", "…but add little to AP")
    for ax in axes:
        style(ax)
    fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc="upper left", ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    save(fig, "r9_external_relations")

In [ ]:
# CELL F16 — draw every result figure and list the outputs
for fn in (r_seed_distribution, r_pr_curves, r_gnn_minus_pair, r_task1_seeds, r_coldstart_community, r_known_market,
           r_attribution, r_portfolios, r_external):
    figure(fn)
made = sorted(p.name for p in OUT.glob("*.pdf"))
print(f"\n{len(made)} figures in {OUT}:")
for m in made:
    print("  ", m)

  saved r1_task2_seed_distribution
  saved r2_precision_recall
  saved r3_gnn_minus_pair
  saved r4_task1_seeds
  saved r5_coldstart_community
  saved r6_known_market
  saved r7_feature_attribution
  saved r8_portfolios
  saved r9_external_relations

16 figures in /content/drive/MyDrive/KG_Strong_Pipeline/figures_q1:
   kg1_ego_network.pdf
   kg2_community_metagraph.pdf
   kg3_relation_multiplex.pdf
   kg4_class_connectivity.pdf
   kg5_degree_ccdf.pdf
   kg6_embedding_map.pdf
   kg7_structural_exposure.pdf
   r1_task2_seed_distribution.pdf
   r2_precision_recall.pdf
   r3_gnn_minus_pair.pdf
   r4_task1_seeds.pdf
   r5_coldstart_community.pdf
   r6_known_market.pdf
   r7_feature_attribution.pdf
   r8_portfolios.pdf
   r9_external_relations.pdf
